# Steered Gaussian Derivative

This uses a first derivative of Gaussian to compute `gradient_x` and `gradient_y`, then steers those two outputs to any angle.

## How It Works

A first-order Gaussian derivative is steerable because any directional derivative is a linear combination of the horizontal and vertical derivatives. The steering equation is `response = cos(theta) * gradient_x + sin(theta) * gradient_y`.

### Reference

Freeman and Adelson introduced steerable filters in 1991. DOI [10.1109/34.93808](https://doi.org/10.1109/34.93808). Canny's 1986 edge detector is a common first-derivative Gaussian reference. DOI [10.1109/TPAMI.1986.4767851](https://doi.org/10.1109/TPAMI.1986.4767851).

## Setup

These notebooks are split by how the orientation response is made. This one first computes `gradient_x` and `gradient_y`, then projects that gradient pair onto an angle.


In [ ]:
# Imports.
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import torch

from agfb_filters import (
    ExecutionPath,
    get_filter_definition,
    orientation_angles,
    run_filter,
    steer_gradient,
)

## Filter Settings

Change `sigma`, `angle_count`, or `angle_index` below to see how the response changes.

In [ ]:
# Choose the filter and angle grid.
sigma = 5.0
angle_count = 12
definition = get_filter_definition("derivative_of_gaussian", sigma=sigma)
path = ExecutionPath.SEPARABLE
angles = orientation_angles(angle_count)

## Test Image

The default image is a 1024 by 1024 black-to-white step edge. The commented lines show the smallest path-based image import you can use instead.


In [ ]:
# Build the test image.
size = 1024
image = torch.zeros(1, size, size)
image[:, :, size // 2 :] = 1.0

# To use your own image instead, uncomment these lines.
# image_path = "path/to/image.png"
# image = torch.as_tensor(plt.imread(image_path), dtype=torch.float32)
# if image.ndim == 3:
#     image = image[..., :3].mean(dim=-1)
# image = image.unsqueeze(0)

plt.figure(figsize=(4, 4))
plt.imshow(image[0], cmap="gray", vmin=0, vmax=1)
plt.title("input image")
plt.axis("off")

## Run And Plot

The filter returns `gradient_x` and `gradient_y`. The last line builds one oriented response from those two arrays. Change `angle_index` to look at a different direction.


In [ ]:
# Time the gradient call and steer one orientation.
start = time.perf_counter()
gradient_x, gradient_y = run_filter(
    definition, image, path=path, boundary=definition.default_boundary
)
result = steer_gradient(gradient_x, gradient_y, angles, definition_name=definition.name)
elapsed = time.perf_counter() - start
print(f"{elapsed:.4f} seconds")

angle_index = 0
response = result.responses[0, angle_index]
limit = max(
    float(gradient_x.abs().max()), float(gradient_y.abs().max()), float(response.abs().max())
)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(gradient_x[0], cmap="gray", vmin=-limit, vmax=limit)
axes[0].set_title("gradient_x")
axes[1].imshow(gradient_y[0], cmap="gray", vmin=-limit, vmax=limit)
axes[1].set_title("gradient_y")
axes[2].imshow(response, cmap="gray", vmin=-limit, vmax=limit)
axes[2].set_title(f"angle {float(result.angles[angle_index]):.2f}")
for axis in axes:
    axis.axis("off")
plt.tight_layout()